# Recap: Hierarchical Clustering


<!--
In-class recap of the pre-recorded lecture 04-Clustering-II-hierarchical.qmd.

NO `jupyter: python3` and no executable code blocks — markdown, LaTeX and static
figs/ images only, so this deck adds zero freeze/cache burden.

Cold-call breakpoints are the invisible `cold-call:` comments below; IDs live in
instructor/question-banks/04-Clustering-II-hierarchical.md. Knowledge-check
questions are 04-KC01..04-KC03 in instructor/knowledge-checks/04-Clustering-II-hierarchical.md.

Load-bearing ideas this deck reinforces (not a tour of the lecture):
  1. the agglomerative procedure and what a dendrogram encodes;
  2. the four linkage criteria and their characteristic behaviours;
  3. hierarchical vs partitional — what you get when you do not know k.
-->

## Today's plan

- 5 min — knowledge-check review
- 20 min — highlights and Q&A (answer or pass — answering always earns credit)
- 60 min — in-class activity (small groups)
- wrap-up and cold-call check-ins


# Knowledge-Check Review

## KC 1: The procedure and the picture

*Describe the agglomerative hierarchical clustering procedure in three steps, and say what the height at which two branches join in a dendrogram represents.*

<!-- cold-call: 04-Q03 -->

:::: {.kc-answer}
**Answer.**

1. Start with **every point as its own cluster**.
2. **Merge the two closest clusters** — "closest" as defined by the *linkage* (single, complete, average, Ward, …).
3. **Repeat** until one cluster remains.

The **height of a join** is the inter-cluster distance at the moment those two clusters were merged. Low joins are tight groups, tall joins are far-apart groups — and a horizontal cut at any height reads off one flat clustering.

::::

## KC 2: min versus max

*Single linkage defines the distance between two clusters as the distance between their **closest** pair of points; complete linkage uses their **farthest** pair. Explain, from these definitions alone, why single linkage tends to produce long chains that can connect two well-separated groups through a few noise points, and why complete linkage tends toward compact clusters of similar diameter.*

<!-- cold-call: 04-Q07 -->

:::: {.kc-answer}
**Single** — one close cross pair is *enough* to merge, so a cluster grows by absorbing whatever is nearest to *any* member. A string of noise points, each close to the next, stitches two distant groups together: **chaining**. (Same property lets it follow a crescent.)


**Complete** — a merge is charged the *farthest* cross pair, i.e. the **diameter** of the merged cluster. Merging anything elongated or already large is expensive, so clusters stay compact and similar in size; a thin bridge of points does not shrink the max — but one far outlier inflates it.

::::

## KC 3: When the client will not tell you $k$

*A client asks you to segment their customers but cannot tell you how many segments they want. Explain why hierarchical clustering is a natural fit here compared to $k$-means, and how you would still get a concrete number of segments out of it.*

<!-- cold-call: 04-Q11 -->

:::: {.kc-answer}
**Why hierarchical:** $k$-means needs $k$ *before* it runs and returns one partition. Hierarchical clustering does not decide $k$ — the dendrogram holds a nested clustering at **every scale**, so all candidate segmentations are visible at once (and the tree may itself be meaningful, like a taxonomy).


**Getting a number anyway:** cut the tree —

- at a height where there is a **large gap** between successive merges,
- or flatten to $k = 2, 3, \dots$ with `fcluster` and **score** each cut (silhouette — first peak, as on the newsgroups),
- or at the granularity the business can **act on**.

::::

## KC 4: Manually Build H-Cluster

*On the figures in the [exercise](https://ds701.cds.bu.edu/04-Clustering-II-hierarchical.html#exercise) of the lecture draw out the clusters formed via an agglomerative (bottoms-up) hierarchical clustering process. Draw out the corresponding dendrogram as well. You can just visually assess the distances to pick the clusters and approximate the heights on the dendrogram.*

In [ ]:
#| fig-align: center
#| echo: false
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

# Create 5 points with labels (same as before)
points = {
    'A': (3.4, 6.1),
    'B': (1.8, 2.3),
    'C': (4.6, 4.9),
    'D': (1.6, 5.6),
    'E': (2.9, 1.7)
}

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 4.5))

# First subplot: Scatter plot with clusters
for label, (x, y) in points.items():
    ax1.scatter(x, y, s=100, alpha=0.7)
    ax1.annotate(label, (x, y), xytext=(5, 5), textcoords='offset points', 
                fontsize=12, fontweight='bold')


ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Clusters at Different Levels')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 7)
ax1.set_ylim(0, 7)

# Create empty axis with 5 evenly spaced ticks
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 1)
ax2.set_xticks([0, 1, 2, 3, 4])
ax2.set_xticklabels([])
ax2.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax2.set_yticklabels([])
ax2.set_title('Dendrogram')
ax2.set_xlabel('Points')
ax2.set_ylabel('Distance')

plt.tight_layout()

# Save the plot as a png image (uncomment to save)
# plt.savefig('figs/L08-dendrogram.png')
plt.show()

# Highlights

## Not one partition — a whole tree

Last session: a **strict partition** — every point in exactly one of $k$ clusters, $k$ chosen in advance.

* But "how many clusters?" often has several honest answers, depending on **scale**.
* A hierarchical clustering returns **nested** clusters organised as a tree.
* The **dendrogram** records the containment relations: leaves are points, each join is a merge, height is the distance at which it happened.
* It **does not decide** the number of clusters — you cut it, at any level, afterwards.

![](figs/L08-dendrogram.png)


<!-- cold-call: 04-Q01 -->
<!-- cold-call: 04-Q02 -->

## The agglomerative procedure

1. **Initialise**: each point is its own cluster; compute the pairwise distance matrix if not given.
2. **Merge** the two closest clusters (per the linkage and the metric).
3. **Update** the distances from the new cluster to all remaining clusters.
4. **Repeat** 2–3 until one cluster remains.

The output is a **linkage matrix** of shape $(n-1, 4)$ — one row per merge:

$$
[\ \text{idx}_1,\ \text{idx}_2,\ \text{distance},\ \text{count}\ ]
$$

Indices $0 \dots n-1$ are original points; indices $n, n+1, \dots$ are the clusters created by earlier rows. `scipy.cluster.hierarchy.linkage` produces it; `dendrogram` draws it; `fcluster` cuts it.


<!-- cold-call: 04-Q04 -->

## Everything hinges on the linkage

Distance between two *clusters* $C_i, C_j$ — three natural choices, plus one borrowed from $k$-means:

![](figs/L08-hierarchical-criteria-a.jpeg)

**Single**
$$\min_{x\in C_i,\,y\in C_j} d(x,y)$$

![](figs/L08-hierarchical-criteria-b.jpeg)

**Complete**
$$\max_{x\in C_i,\,y\in C_j} d(x,y)$$

![](figs/L08-hierarchical-criteria-c.jpeg)

**Average**
$$\frac{1}{|C_i||C_j|}\sum_{x\in C_i,\,y\in C_j} d(x,y)$$


**Ward**: the *increase* in within-cluster sum of squares if $C_i$ and $C_j$ were merged — merge the pair that hurts the $k$-means objective least. A hierarchical, greedy $k$-means.


<!-- cold-call: 04-Q05 -->
<!-- cold-call: 04-Q09 -->

## Same data, four personalities

| linkage | good at | fails when | shape bias |
|---|---|---|---|
| **single** | connected, odd shapes (moons, rings); unequal sizes | a few noise points bridge two groups → **chaining** | none |
| **complete** | balanced, similar-diameter clusters; ignores bridges | one far **outlier** dominates the max | compact / spherical |
| **average** | a compromise; less noise- and outlier-sensitive | — | elliptical |
| **ward** | like $k$-means: robust, tidy blobs | non-convex shapes; needs Euclidean features | elliptical |

## Comparison of linkages

No linkage wins every row — the linkage is a **modelling choice** about what a cluster should look like.

[Scikit-Learn Comparison](https://scikit-learn.org/stable/auto_examples/cluster/plot_linkage_comparison.html)

![](figs/L08-sphx_glr_plot_linkage_comparison_001.png)

<!-- cold-call: 04-Q06 -->
<!-- cold-call: 04-Q08 -->
<!-- cold-call: 04-Q12 -->

## Cutting the tree {.shrink style="--shrink: 0.9; --shrink-title: 1"}

![](figs/L08-dendrogram-cut.png)

Two ways to flatten a hierarchy:

* **by height** — `fcluster(Z, h, criterion="distance")`: every merge below $h$ is done. The red and green lines are two such cuts.
* **by count** — `fcluster(Z, k, criterion="maxclust")`: at most $k$ clusters.
* Where to cut? Look for a **large gap** between successive merge heights, or sweep $k$ and score each cut (silhouette), or ask what the clusters are *for*.


Heights are only comparable **within one dendrogram** — single-linkage heights are nearest-neighbour distances, Ward's accumulate squared error.


<!-- cold-call: 04-Q11 -->

## Hierarchical vs. partitional {.shrink style="--shrink: 0.9; --shrink-title: 1"}

| | $k$-means (partitional) | agglomerative (hierarchical) |
|---|---|---|
| needs $k$ up front | **yes** | no — cut afterwards |
| output | one partition | a tree of nested partitions |
| input | points in $\mathbb{R}^d$ (needs means) | feature matrix **or** any distance / similarity matrix |
| cluster shape | spherical Voronoi cells | depends on linkage |
| cost | cheap, iterative | $O(n^2)$ memory, $O(n^2 \log n)$ time — fine for thousands, not millions |
| repeatability | random init | deterministic |

Reach for a dendrogram when you **do not know $k$**, when the **nesting itself is the answer** (taxonomies), or when all you have is a **distance matrix**. 

Today's activity: the same data, both tools, and three kinds of mess — find out for yourself which survives what.


<!-- cold-call: 04-Q10 -->

## Going deeper

Full lecture notes: [Hierarchical Clustering](04-Clustering-II-hierarchical.qmd)

- [How Many Clusters?](04-Clustering-II-hierarchical.qmd#how-many-clusters) — the same data at 3, 4, 5 and 6 clusters
- [Example: Dendrogram](04-Clustering-II-hierarchical.qmd#example-dendrogram) — five points, built up merge by merge
- [Hierarchical Clustering Algorithms](04-Clustering-II-hierarchical.qmd#hierarchical-clustering-algorithms-1) — agglomerative vs divisive; the linkage matrix
- [Defining Cluster Proximity](04-Clustering-II-hierarchical.qmd#defining-cluster-proximity) — single, complete, average, and their strengths and failures on moons and blobs
- [Ward's Distance](04-Clustering-II-hierarchical.qmd#wards-distance)
- [Selecting the Number of Clusters](04-Clustering-II-hierarchical.qmd#selecting-the-number-of-clusters) — silhouette on the newsgroups tree
- [Comparison of Linkages](04-Clustering-II-hierarchical.qmd#comparison-of-linkages) — scikit-learn's side-by-side figure
- Cluster evaluation (ARI, silhouette) and real-data practice: [Clustering in Practice](M10-Clustering-in-Practice.qmd)


# In-Class Activity

## Activity: dendrograms on real data, then make it messy

Goal: build and *read* dendrograms for four linkages on a real dataset, cut them into flat clusters, then inject a unit change, two outliers and a chaining bridge and watch which linkages survive — and whether $k$-means would have been fine all along.

- Work in groups of 2–3. Open **your section's** notebook.
- Parts marked **(autograded)** are submitted to Gradescope; the rest is participation.
- Staff will circulate — be ready to explain any part of your work.
- Dependencies: `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/04-InClass-Exercise-Hierarchical-A1/04-InClass-Exercise-Hierarchical-A1.ipynb)
[![](https://img.shields.io/badge/Open%20on-GitHub-181717?logo=github)](https://github.com/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/04-InClass-Exercise-Hierarchical-A1/04-InClass-Exercise-Hierarchical-A1.ipynb)

**Note**

The activity notebook goes live on the day of the lecture. Colab is optional — you can also open it on GitHub and run it locally.
